# Datamine Grade Estimation and Kriging Workflow

This notebook demonstrates the standard workflows for block model prototyping and grade interpolation in Datamine Studio RM using `dmstudio`.

It specifically showcases three processes that are not used in `Holes3D_Tutorial.ipynb`:
1. **`protom`** - Block model prototype creation
2. **`estima`** - Univariate grade estimation (Nearest Neighbour, Inverse Distance, Ordinary Kriging)
3. **`cokrig`** - Advanced multivariate and Ordinary Kriging estimation

In [1]:
# Step 1: Connect to Datamine Studio RM COM Automation
import os
import shutil
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, special, agent

# Connect to the running Datamine Studio RM COM Application
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")
# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
import os
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

Connected to: Studio RM 3.1.390.0 - Project.rmproj * - [3D]


## Step 2: Block Model Prototyping
Before grade estimation can take place, a block model prototype must be initialized.
The **`protom`** process is used to define coordinates limits and cell sizes.
Since the drillhole coordinates in `comp_holes` range from:
- X: 5923 to 6119
- Y: 4821 to 5234
- Z: -98 to 199

We define the block model prototype to cover this volume with cell sizes 10x10x10 and origin 5900, 4800, -100:

In [2]:
# Generate block model prototype
# We pass the interactive console answers (Mined: N, Subcells: Y, Origin: 5900, 4800, -100, Size: 10,10,10, Counts: 25,45,32) via arguments:
print("Generating block model prototype (PROTOM)...")
dm_fil.protom(
    out_o='model_proto',
    rotmod_p=0,
    arguments=" 'N' 'Y' '5900' '4800' '-100' '10' '10' '10' '25' '45' '32'"
)
print("Prototype 'model_proto' created successfully.")

Generating block model prototype (PROTOM)...
Prototype 'model_proto' created successfully.


## Step 3: Copy and Prepare Parameter Files
Next, we prepare the search, estimation, and variogram model files using standard tutorial resources.
For `COKRIG`, we dynamically create fields and parameter tables using `pandas` and `special.inpfil`.

In [3]:
# Locate help database VBOP directory containing standard tutorial parameter files
project_folder = oScript.ActiveProject.Folder
repo_root = os.path.abspath(os.path.join(project_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

print("Copying ESTIMA parameter files...")
shutil.copy(os.path.join(help_db, "_vb_spar.dmx"), os.path.join(project_folder, "search_params.dmx"))
shutil.copy(os.path.join(help_db, "_vb_epar.dmx"), os.path.join(project_folder, "estimation_params.dmx"))

print("Generating COKRIG-compatible parameter files...")

# 1. Create kriging_vmodel by adding GRADE, GRADE2, VSETNUM to _vb_vpar
vpar_src = agent.read_datamine(os.path.join(help_db, "_vb_vpar.dmx"))
vpar_src['GRADE'] = ['CU', 'AU', 'DENSITY']
vpar_src['GRADE2'] = ['CU', 'AU', 'DENSITY']
vpar_src['VSETNUM'] = [2.0, 1.0, 3.0]
vpar_src.to_csv('kriging_vmodel.csv', index=False)
special.inpfil(csv='kriging_vmodel.csv', out_o='kriging_vmodel')

# 2. Create kriging_epar
epar_df = pd.DataFrame({
    'EREFNUM': [1.0, 2.0, 3.0],
    'VSETNUM': [1.0, 2.0, 3.0],
    'SREFNUM': [1.0, 1.0, 2.0],
    'IMETHOD': [3.0, 3.0, 3.0],
    'DISCX': [3.0, 3.0, 3.0],
    'DISCY': [3.0, 3.0, 3.0],
    'DISCZ': [3.0, 3.0, 3.0],
    'PARENT': [1.0, 1.0, 1.0]
})
epar_df.to_csv('kriging_epar.csv', index=False)
special.inpfil(csv='kriging_epar.csv', out_o='kriging_epar')

# 3. Create kriging_fields
fields_df = pd.DataFrame({
    'EREFNUM': [1.0, 2.0, 3.0],
    'IN_VAR': ['AU', 'CU', 'DENSITY'],
    'EST': ['AU_EST', 'CU_EST', 'DENS_EST'],
    'VAR': ['AU_VAR', 'CU_VAR', 'DENS_VAR'],
    'NUMSAMP': ['AU_NS', 'CU_NS', 'DENS_NS']
})
fields_df.to_csv('kriging_fields.csv', index=False)
special.inpfil(csv='kriging_fields.csv', out_o='kriging_fields')

# Clean up local temporary CSVs
for f in ['kriging_vmodel.csv', 'kriging_epar.csv', 'kriging_fields.csv']:
    if os.path.exists(f):
        os.remove(f)

print("Parameter files setup complete.")

Copying ESTIMA parameter files...
Generating COKRIG-compatible parameter files...
Parameter files setup complete.


## Step 4: Univariate Grade Estimation (ESTIMA)
The **`estima`** process interpolates grade values from drillholes into the block model prototype using methods like Inverse Distance Weighting (IDW) or Nearest Neighbour (NN).

In [4]:
# Run univariate grade estimation using ESTIMA
# Safely clean up any existing file first
estima_file = os.path.join(project_folder, 'grade_model_estima_result.dmx')
if os.path.exists(estima_file):
    try:
        os.remove(estima_file)
    except PermissionError:
        print("Warning: grade_model_estima_result.dmx is locked by Datamine Studio RM. Please unload it in the UI or estimation will fail.")

print("Running univariate grade estimation (ESTIMA)...")
dm_cmd.estima(
    proto_i='model_proto',
    in_i='comp_holes',
    srcparm_i='search_params',
    estparm_i='estimation_params',
    vmodparm_i='kriging_vmodel',
    model_o='grade_model_estima_result',
    x_f='X',
    y_f='Y',
    z_f='Z'
)
print("ESTIMA completed successfully.")

Running univariate grade estimation (ESTIMA)...
ESTIMA completed successfully.


## Step 5: Advanced Multivariate Kriging (COKRIG)
For more advanced multivariate geostatistics and parallel-processed Ordinary/Simple Kriging, the **`cokrig`** process is used.
This function is also fully wrapped inside `dmcommands`.

In [5]:
# Run advanced Kriging using COKRIG
# Safely clean up any existing file first
cokrig_file = os.path.join(project_folder, 'grade_model_cokrig_result.dmx')
if os.path.exists(cokrig_file):
    try:
        os.remove(cokrig_file)
    except PermissionError:
        print("Warning: grade_model_cokrig_result.dmx is locked by Datamine Studio RM. Please unload it in the UI or estimation will fail.")

print("Running advanced kriging estimation (COKRIG)...")
try:
    dm_cmd.cokrig(
        samples_i='comp_holes',
        proto_i='model_proto',
        fields_i='kriging_fields',
        epar_i='kriging_epar',
        spar_i='search_params',
        vmodel_i='kriging_vmodel',
        outmodel_o='grade_model_cokrig_result',
        xpt_f='X',
        ypt_f='Y',
        zpt_f='Z'
    )
    print("COKRIG execution finished.")
except Exception as e:
    print("COKRIG execution failed with exception:", e)

Running advanced kriging estimation (COKRIG)...
COKRIG execution finished.


## Step 6: Verify and Load Results in Pandas
Finally, we can load the newly estimated block models into pandas to verify the estimated grades.

In [9]:
# Verify and read ESTIMA output
if os.path.exists(estima_file):
    shutil.copy(estima_file, 'grade_model_estima_temp.dmx')
    df_estima = agent.read_datamine('grade_model_estima_temp.dmx')
    os.remove('grade_model_estima_temp.dmx')
    print(f"ESTIMA block model cells: {len(df_estima)}")
    print("ESTIMA Gold (AU) Grade Summary:")
    print(df_estima[df_estima['AU'] > -99]['AU'].describe())
else:
    print("Error: grade_model_estima_result.dmx was not found!")

# Verify and read COKRIG output
if os.path.exists(cokrig_file):
    shutil.copy(cokrig_file, 'grade_model_cokrig_temp.dmx')
    df_cokrig = agent.read_datamine('grade_model_cokrig_temp.dmx')
    os.remove('grade_model_cokrig_temp.dmx')
    print(f"COKRIG block model cells: {len(df_cokrig)}")
    print("COKRIG Gold (AU_EST) Grade Summary:")
    print(df_cokrig[df_cokrig['AU_EST'] > -99]['AU_EST'].describe())
else:
    print("COKRIG output was not found. (Check if your Datamine Studio RM license includes the Advanced Estimation module).")

ESTIMA block model cells: 36000
ESTIMA Gold (AU) Grade Summary:
count    36000.000000
mean         2.383928
std          1.243853
min         -0.030780
25%          1.248667
50%          2.394113
75%          3.224252
max         11.357555
Name: AU, dtype: float64
COKRIG block model cells: 36000
COKRIG Gold (AU_EST) Grade Summary:
count    36000.000000
mean         2.384049
std          1.241600
min         -0.017793
25%          1.249698
50%          2.394113
75%          3.224252
max         11.231023
Name: AU_EST, dtype: float64
